# Notebook A — Bacterial Growth Dynamics

**Notebook A — High School Internship · Quantitative Microbiology Lab**

This notebook explores how simple rules at the level of individual cells
produce the growth curves observed in the lab.

---

### How to use this notebook
- Run each cell in order: **Shift + Enter**
- Read the boxed questions `>` and think about them before moving on
- Change the parameters and observe what happens

> **Recommended environment**: [Google Colab](https://colab.research.google.com) — no installation needed.
> Locally: `pip install numpy matplotlib pandas`
>
> **Estimated duration**: 4 half-days (sections A1–A4)


## Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 12
print("Libraries loaded.")

---
## Section A1 — Exponential growth: the deterministic model

A bacterium divides in two. Its two daughters divide in two. And so on.

If at each time step every cell has probability $p$ of dividing,
the average number of new divisions is $p \times N$.

**Deterministic model**: we ignore fluctuations and write directly:

$$N(t+1) = N(t) \times (1 + p)$$

This is **repeated multiplication by the same factor** $(1+p)$.

> **Before coding**: if $N_0 = 10$ and $p = 0.1$, what is $N$ after 1 step?
> After 2 steps? Compute the first 3 values by hand.


In [ ]:
# ── Parameters ─────────────────────────────────────────────────────────────────
N0      = 10    # nombre initial de cellules
p       = 0.1   # division probability per time step  (must be << 1)
n_steps = 60    # nombre de pas de temps simulés

# ── Deterministic simulation ────────────────────────────────────────────────────────
N = N0
Ns_det = [N]
for i in range(n_steps):
    N = N * (1 + p)    # multiplication repetee : N(t+1) = N(t) * (1 + p)
    Ns_det.append(N)

# ── Linear scale plot ─────────────────────────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(Ns_det, color='crimson', lw=2)
plt.xlabel("Time step")
plt.ylabel("Number of cells N")
plt.title("Bacterial growth — deterministic model")
plt.tight_layout()
plt.show()

### Logarithmic representation

In **log scale**, repeated multiplication by $(1+p)$ becomes
a **straight line** — which lets you visually identify the exponential growth phase
on your OD curves.

The steeper the slope, the faster the bacteria divide.

> **Question**: what is the relationship between the slope of this line and parameter $p$?
> And with the **doubling time** of the bacteria?


In [ ]:
# Same curve in log scale
plt.figure(figsize=(8, 4))
plt.plot(Ns_det, color='crimson', lw=2)
plt.yscale('log')
plt.xlabel("Time step")
plt.ylabel("Number of cells N  (log scale)")
plt.title("Log scale: repeated multiplication becomes a straight line")
plt.tight_layout()
plt.show()

# Calcul du temps de doublement theorique
# (np.log est la fonction logarithme — elle sera etudiee en terminale ;
#  pour l'instant, verifiez graphiquement à quel pas N a double par rapport a N0)
t_doublement = np.log(2) / np.log(1 + p)
print(f"Temps de doublement : {t_doublement:.1f} pas")
print(f"Valeur finale       : N = {Ns_det[-1]:.0f} cellules apres {n_steps} pas")

---
## Section A2 — Stochastic simulation: randomness cell by cell

The deterministic model gives the **expected average value**.
But in reality, each cell decides to divide or not in a **random** way.

**Microscopic rule**: at each time step, for *each* individual cell,
we draw at random — it divides with probability $p$, independently of its neighbours.

> **Question**: if chance plays a role at each cell,
> why do we obtain a smooth curve at the population level?

We will simulate this directly and observe what happens.

---

> ⚠️ **Technical note**: this simulation assumes $p \ll 1$ (small probability per time step).
> This is the so-called *τ-leaping* approximation, a discrete version of the Gillespie algorithm.
> For $p = 0.1$, the approximation is very good.


In [ ]:
def simulate_one(N0, p, n_steps):
    """Simulates one stochastic growth trajectory.

    At each step, each cell draws: it divides (probability p) or not.
    Assumption: p << 1 (tau-leaping approximation).
    """
    N = N0
    Ns = [N]
    for _ in range(n_steps):
        # np.random.random(N) génère N tirages uniformes dans [0, 1)
        # La cellule divise si son tirage est inférieur à p
        divisions = np.sum(np.random.random(N) < p)
        N += divisions
        Ns.append(N)
    return Ns

# Test with a single trajectory
Ns_stoch = simulate_one(N0=10, p=0.1, n_steps=60)

plt.figure(figsize=(8, 4))
plt.plot(Ns_det,   'r-',  lw=2.5, label='Deterministic')
plt.plot(Ns_stoch, 'b-',  alpha=0.8, label='Stochastic (1 trajectory)')
# plt.yscale('log')
plt.xlabel("Time step")
plt.ylabel("Number of cells")
plt.title("Deterministic vs one stochastic trajectory")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
n_traj   = 30
all_traj = []

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for _ in range(n_traj):
    traj = simulate_one(N0=10, p=0.1, n_steps=60)
    all_traj.append(traj)
    for ax in axes:
        ax.plot(traj, alpha=0.15, color='steelblue', lw=1)

mean_traj = np.mean(all_traj, axis=0)

for ax, scale, titre in zip(axes,
                            ['linear', 'log'],
                            ['Échelle linéaire', 'Échelle logarithmique']):
    ax.plot(Ns_det,    'r-',  lw=2.5, label='Deterministic')
    ax.plot(mean_traj, 'k--', lw=2,   label='Stochastic mean')
    ax.set_yscale(scale)
    ax.set_xlabel("Time step")
    ax.set_ylabel("Number of cells")
    ax.set_title(titre)
    ax.legend()

plt.suptitle(f"{n_traj} independent stochastic trajectories", fontsize=13)
plt.tight_layout()
plt.show()

> **Observation**: each trajectory is different, but their **mean** follows
> the deterministic curve closely. In log scale, trajectories stay
> clustered around a straight line.
>
> This is the **law of large numbers**: random individual rules
> produce predictable population-level behaviour.
>
> **To explore**: re-run the simulation with $N_0 = 2$ instead of $N_0 = 10$.
> Are the trajectories more or less spread? Why?


---
## Section A3 — Nutrient depletion and stationary phase

In a closed medium (tube, microwell plate), nutrients are depleted as
bacteria multiply. Growth eventually stops.

**Idea**: the division probability $p$ is no longer constant —
it depends on the available nutrient concentration $S$.

### The Monod curve

$$p(S) = p_{\max} \times \frac{S}{K_S + S}$$

| Situation | Value of $p(S)$ | Interpretation |
|-----------|:---------------:|----------------|
| $S \gg K_S$ | $\approx p_{\max}$ | Abundant nutrients → maximum growth |
| $S = K_S$ | $p_{\max}/2$ | $K_S$ sets the saturation scale |
| $S \ll K_S$ | $\approx 0$ | Nutrients exhausted → growth stops |

This form is identical to Michaelis-Menten enzyme kinetics.


In [ ]:
# Visualise the Monod curve before using it in the simulation
p_max = 0.1     # maximum division probability (abundant nutrients)
Ks    = 20.0    # half-saturation: S at which p = p_max / 2

S_vals = np.linspace(0, 200, 400)
p_vals = p_max * S_vals / (Ks + S_vals)

plt.figure(figsize=(7, 4))
plt.plot(S_vals, p_vals, color='darkgreen', lw=2.5)
plt.axhline(p_max,     color='gray', linestyle='--', alpha=0.8, label=f'p_max = {p_max}')
plt.axhline(p_max / 2, color='gray', linestyle=':',  alpha=0.5, label='p_max / 2')
plt.axvline(Ks,        color='red',  linestyle='--', alpha=0.8, label=f'Ks = {Ks}')
plt.xlabel("Nutrient concentration S")
plt.ylabel("Division probability p(S)")
plt.title("Monod curve: growth saturates as nutrients deplete")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def simulate_nutrients(N0, S0, p_max, Ks, Y, n_steps):
    """Simulates bacterial growth with nutrient depletion (Monod model).

    Parameters
    ----------
    N0     : initial number of cells
    S0     : initial nutrient quantity
    p_max  : maximum division probability (abundant nutrients)
    Ks     : half-saturation constant (S at which p = p_max / 2)
    Y      : yield (cells produced per unit of nutrient consumed)
    n_steps: number of time steps
    """
    N, S = float(N0), float(S0)
    Ns, Ss = [N], [S]
    for _ in range(n_steps):
        p = p_max * S / (Ks + S)                    # Monod : p dépend des nutriments
        divisions = np.sum(np.random.random(int(N)) < p)
        N += divisions
        S -= divisions / Y                           # chaque division consomme 1/Y
        S  = max(S, 0.0)                             # S ne peut pas être négatif
        Ns.append(N)
        Ss.append(S)
    return Ns, Ss

# Parameters
N0_m, S0_m            = 10, 500
p_max_m, Ks_m, Y_m   = 0.1, 20.0, 5.0
n_m                   = 200

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

for _ in range(15):
    Ns_m, Ss_m = simulate_nutrients(N0_m, S0_m, p_max_m, Ks_m, Y_m, n_m)
    axes[0].plot(Ns_m, alpha=0.2, color='steelblue',  lw=1)
    axes[1].plot(Ss_m, alpha=0.2, color='darkorange', lw=1)

axes[0].set_yscale('log')
axes[0].set_ylabel("Number of cells N")
axes[0].set_title("Bacterial growth with nutrient depletion (Monod model)")
axes[1].set_ylabel("Nutrients S")
axes[1].set_xlabel("Time step")
plt.tight_layout()
plt.show()

### Carrying capacity prediction

In our model, each division consumes exactly $1/Y$ units of nutrients
($Y$ is the **yield**: number of cells produced per unit of nutrient consumed).

Starting from $S_0$ nutrients and $N_0$ cells, when all nutrients are exhausted:

$$N_{\max} = N_0 + Y \times S_0$$

**This is not a fitted parameter — it is a model prediction.**
The next cell will verify this prediction numerically.


In [ ]:
# Carrying capacity is a MODEL PREDICTION, not a fitted parameter
N_max_pred = N0_m + Y_m * S0_m
print(f"Theoretical prediction: N_max = N0 + Y * S0 = {N0_m:.0f} + {Y_m:.0f} * {S0_m:.0f} = {N_max_pred:.0f}")

# Numerical check: final value over 50 independent simulations
N_finals = [simulate_nutrients(N0_m, S0_m, p_max_m, Ks_m, Y_m, n_m)[0][-1]
            for _ in range(50)]
print(f"Valeur simulée     : {np.mean(N_finals):.0f} ± {np.std(N_finals):.0f}  (n = 50 trajectoires)")
print()
print("These two values should be close if the model is consistent!")

> **Note**: the simulation typically gives ~1% more than the prediction
> $N_{\max} = N_0 + Y \cdot S_0$. The prediction assumes a sharp cut-off
> when $S = 0$, but in the discrete model $p(S) \to 0$ gradually —
> a few divisions still occur when $S$ is very small.
> This is an expected discrepancy, not a bug.


---
## Section A4 — Comparison with experimental data

### First test: is biomass produced proportional to glucose?

The Monod model predicts $N_{\max} = N_0 + Y \times S_0$,
i.e. a **linear** relationship between biomass produced and initial glucose,
with a line passing through the origin.

We can verify this directly with your Tuesday data:
for each tube, we compute $\Delta\text{OD} = \text{OD}_{\max} - \text{OD}_0$.
If the model is correct, the three points should be aligned on a line through zero.

The **yield** $Y$ is the slope of this line:
how many OD units of bacteria are produced per gram of glucose consumed.


In [ ]:
# ── Entrez vos valeurs mesurées mardi ──────────────────────────────────────────
glucose_gL    = [0.2, 2.0, 4.0]      # concentrations en g/L (0.02%, 0.2%, 0.4%)
OD_initial    = 0.005                # OD au temps 0 (meme pour les 3 tubes)
OD_max_mesure = [0.15, 1.5, 3.2]     # à remplir avec vos valeurs lues sur les courbes

# ── Biomasse produite ──────────────────────────────────────────────────────────
delta_OD = [od - OD_initial for od in OD_max_mesure]

for glu, dod in zip(glucose_gL, delta_OD):
    print(f"  [glucose] = {glu:.1f} g/L  |  delta_OD = {dod:.3f}")

# ── Estimation de Y : pente de chaque point, puis moyenne ─────────────────────
# Each point gives a Y estimate = delta_OD / [glucose]
Y_vals = [dod / glu for dod, glu in zip(delta_OD, glucose_gL)]
Y_fit  = sum(Y_vals) / len(Y_vals)

print(f"
Yield Y estimated per condition:")
for glu, y in zip(glucose_gL, Y_vals):
    print(f"  [glucose] = {glu:.1f} g/L  ->  Y = {y:.3f} OD/(g/L)")
print(f"Moyenne : Y = {Y_fit:.3f} OD/(g/L)")

# ── Plot ─────────────────────────────────────────────────────────────────────────
glu_dense = np.linspace(0, max(glucose_gL) * 1.1, 100)

plt.figure(figsize=(7, 4))
plt.plot(glucose_gL, delta_OD, 'ko', ms=10, zorder=4, label='Data (Tuesday)')
plt.plot(glu_dense, [Y_fit * g for g in glu_dense], 'b--', lw=1.5,
         label=f'y = Y·x  (Y = {Y_fit:.3f} OD/(g/L))')
plt.xlabel("Initial [glucose] (g/L)")
plt.ylabel("Biomass produced  (OD_max - OD_0)")
plt.title("Validation of N_max = N0 + Y × S0")
plt.legend()
plt.tight_layout()
plt.show()

# ── Conversion en cellules ──────────────────────────────────────────────────────
cells_per_OD = 2e9    # ~2×10^9 cellules par unite OD pour E. coli
print(f"
Conversion (1 OD ~ {cells_per_OD:.0e} cells):")
print(f"  Y = {Y_fit:.3f} OD/(g/L)  ≈  {Y_fit * cells_per_OD:.2e} cellules/(g/L)")
print()
print("-> Are the three points aligned on the line?")
print("   Is Y approximately the same for all 3 concentrations?")
print("   If not: which model assumption might be wrong?")

### Reading plate reader data
The plate reader measured OD600 of your cultures every few minutes
throughout the night. We will load these data to:
1. verify biomass produced as a function of nutrient concentration for different glucose and lactose conditions.
2. measure the growth rate in these different conditions.
3. visualise the diauxie growth curves.

#### BioTek Gen5 file format

Same format as in the induction notebook: two TSV blocks (OD + GFP),
with the channel name as the first line of each block.
The `parse_biotek` function below handles this format.

> **Note**: if you measured growth curves manually (OD on spectrophotometer),
> you can load a simple CSV with `pd.read_csv()`:
> column `temps_h` (hours) + one column per tube.


In [ ]:
import pandas as pd

def parse_biotek(filepath):
    # Reads a BioTek Gen5 plate reader file (multi-block TSV format).
    # Returns a dict {channel_name : DataFrame}
    # Each DataFrame: 'temps_h' column (decimal hours) + columns A1...H12.
    with open(filepath, encoding='utf-8', errors='replace') as f:
        lines = f.read().splitlines()
    block_starts = []
    for i, line in enumerate(lines):
        s = line.strip()
        if s and chr(9) not in s and not s[0].isdigit():
            block_starts.append(i)
    result = {}
    for b_idx, b_start in enumerate(block_starts):
        canal_name = lines[b_start].strip()
        if b_start + 2 >= len(lines):
            continue
        header = lines[b_start + 2].strip().split(chr(9))
        b_end = block_starts[b_idx+1] if b_idx+1 < len(block_starts) else len(lines)
        rows = []
        for line in lines[b_start + 3 : b_end]:
            parts = line.strip().split(chr(9))
            if len(parts) < 3:
                continue
            t_str = parts[0]
            if not t_str or t_str == '0:00:00':
                continue
            t_parts = t_str.split(':')
            if len(t_parts) != 3:
                continue
            try:
                h, m, s = int(t_parts[0]), int(t_parts[1]), int(t_parts[2])
            except ValueError:
                continue
            rows.append([h + m/60 + s/3600] + parts[2:])
        if not rows:
            continue
        wells = header[2:]
        n_cols = len(rows[0]) - 1
        cols = ['temps_h'] + wells[:n_cols]
        df = pd.DataFrame(rows, columns=cols)
        for col in df.columns[1:]:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        result[canal_name] = df
    return result


# ══════════════════════════════════════════════════════════════════════════════════
# To load YOUR BioTek data, uncomment:
# ══════════════════════════════════════════════════════════════════════════════════
# data  = parse_biotek('your_diauxie.txt')
# od_df = next(df for name, df in data.items() if '600' in name)
#
# For a manual CSV (time + one column per tube):
# od_df = pd.read_csv('manual_growth.csv').rename(columns={'time': 'temps_h'})

# ── Synthetic diauxie data — REPLACE with your real data ─────────────────────────
np.random.seed(3)

def generer_diauxie():
    # Simulates BioTek curves from an overnight diauxie experiment.
    # Wells A1-A3: glucose only (3 concentrations)
    # Wells B1-B2: glucose + lactose (diauxie)
    # Well H12  : medium only (blank)
    n_pts = 72
    temps = np.linspace(0, 14, n_pts)

    def logistic(t, K, t_mid, mu):
        return K / (1 + np.exp(-mu * (t - t_mid)))

    od = {'temps_h': temps}
    # Glucose only: 3 concentrations → 3 different carrying capacities
    for label, K in [('A1', 0.35), ('A2', 0.80), ('A3', 1.40)]:
        vals = 0.05 + logistic(temps, K, K*3.8, 1.8)
        od[label] = np.clip(vals + np.random.normal(0, 0.008, n_pts), 0, None)
    # Glucose + Lactose: diauxie (two phases with intermediate dip)
    for label in ['B1', 'B2']:
        phase1 = logistic(temps, 0.45, 2.5, 2.2)
        # Lactose phase starts after lacZ induction (~5h total lag)
        phase2 = np.array([logistic(t, 0.42, 9.0, 1.6) if t > 5.0 else 0.0
                           for t in temps])
        vals = 0.05 + phase1 + phase2
        od[label] = np.clip(vals + np.random.normal(0, 0.010, n_pts), 0, None)
    # Blank
    od['H12'] = np.clip(0.082 + np.random.normal(0, 0.003, n_pts), 0, None)

    return pd.DataFrame(od)

od_df = generer_diauxie()
print("Data loaded.")
print(f"  {len(od_df)} time points, {len(od_df.columns)-1} wells")
print(f"  Duree : {od_df['temps_h'].min():.1f} – {od_df['temps_h'].max():.1f} h")
print(od_df[['temps_h', 'A1', 'A2', 'B1', 'H12']].head(3).to_string(index=False))

### Biomass / nutrients — verification on the plate reader

The plate reader gives OD600 for **many** conditions simultaneously.
We will first plot all growth curves, extract the maximum OD
from each well, then verify that $\Delta\text{OD} \propto [\text{glucose}]$.

We can also compare the yield $Y$ measured here with the one obtained
Tuesday by hand — and extend it to **lactose-only** wells if the layout allows.


In [ ]:
# ── Layout — ADAPT to your actual plate ─────────────────────────────────────────
wells_glu     = ['A1', 'A2', 'A3']   # glucose only, increasing concentrations
wells_lac     = []                    # lactose only (leave [] if absent)
blanc         = 'H12'                 # milieu seul

glu_gL = [0.2, 2.0, 4.0]            # concentrations glucose en g/L  ADAPTER
lac_gL = []                          # concentrations lactose en g/L  ADAPTER

# ── Subtract blank ────────────────────────────────────────────────────────────────
od_blanc = od_df[blanc].mean() if blanc in od_df.columns else 0.0

# ── Plot growth curves ───────────────────────────────────────────────────────────
plt.figure(figsize=(9, 4))
for well in wells_glu + wells_lac:
    if well in od_df.columns:
        style = '-' if puit in wells_glu else '--'
        plt.plot(od_df['temps_h'], od_df[puit] - od_blanc, style, lw=1.5, label=well)
plt.xlabel("Time (hours)")
plt.ylabel("OD600 (blank-corrected)")
plt.title("Growth curves — plate reader")
plt.legend()
plt.tight_layout()
plt.show()

# ── ΔOD per well: max − initial value ────────────────────────────────────────────
def delta_od_puit(puit):
    od_corr = od_df[puit] - od_blanc
    return od_corr.max() - od_corr.iloc[0]

delta_glu_pr = [delta_od_puit(p) for p in wells_glu if p in od_df.columns]
delta_lac_pr = [delta_od_puit(p) for p in wells_lac if p in od_df.columns]

# ── Yield Y (average of individual slopes) ────────────────────────────────────────
Y_vals_glu_pr = [dod / glu for dod, glu in zip(delta_glu_pr, glu_gL)]
Y_fit_pr = sum(Y_vals_glu_pr) / len(Y_vals_glu_pr)

print("Yield Y estimated (plate reader, glucose):")
for glu, y in zip(glu_gL, Y_vals_glu_pr):
    print(f"  [glucose] = {glu:.1f} g/L  ->  Y = {y:.3f} OD/(g/L)")
print(f"Moyenne : Y = {Y_fit_pr:.3f} OD/(g/L)")
print(f"Y mesuré mardi (manuel) : Y = {Y_fit:.3f} OD/(g/L)")
print()

# ── Plot ΔOD = f([nutrient]) ────────────────────────────────────────────────────
glu_dense = np.linspace(0, max(glu_gL) * 1.1, 100)

plt.figure(figsize=(7, 4))
plt.plot(glu_gL, delta_glu_pr, 'ko',  ms=10, zorder=4, label='Plate reader (glucose)')
plt.plot(glu_gL, delta_OD,     'rs',  ms=10, zorder=4, label='Manuel mardi (glucose)')
if delta_lac_pr:
    plt.plot(lac_gL, delta_lac_pr, 'b^', ms=10, zorder=4, label='Plate reader (lactose)')
plt.plot(glu_dense, [Y_fit_pr * g for g in glu_dense], 'k--', lw=1.5,
         label=f'y = Y·x  (Y = {Y_fit_pr:.3f} OD/(g/L))')
plt.xlabel("Initial [nutrient] (g/L)")
plt.ylabel("Biomass produced  (OD_max - OD_0)")
plt.title("Validation N_max = N0 + Y × S0 — plate reader")
plt.legend()
plt.tight_layout()
plt.show()

print("-> Plate reader Y ≈ manual Y?")
print("   Does lactose give the same yield as glucose?")

### Comparison with the Monod model

Now that the data are loaded, let us directly compare the shape
of a **glucose-only** curve with the Monod simulation.
If the model is consistent, the two normalised curves should overlap:
same exponential → stationary transition.


In [ ]:
# Compare a normalised glucose-only curve with the Monod simulation
col_ref  = wells_glu[1] if len(wells_glu) > 1 and wells_glu[1] in od_df.columns else wells_glu[0]
OD_ref   = od_df[col_ref] - od_blanc
OD_norm  = (OD_ref - OD_ref.min()) / (OD_ref.max() - OD_ref.min())

Ns_c, _ = simulate_nutrients(N0_m, S0_m, p_max_m, Ks_m, Y_m, n_m)
Ns_arr  = np.array(Ns_c, dtype=float)
Ns_norm = (Ns_arr - Ns_arr.min()) / (Ns_arr.max() - Ns_arr.min())
t_sim   = np.linspace(od_df['temps_h'].min(), od_df['temps_h'].max(), len(Ns_norm))

plt.figure(figsize=(9, 5))
plt.plot(od_df['temps_h'], OD_norm, 'ko-', ms=5,
         label=f'Glucose-only data ({col_ref}, normalisées)')
plt.plot(t_sim, Ns_norm, 'b-', lw=2, alpha=0.8, label='Simulation Monod (normalisée)')
plt.xlabel("Time (hours)")
plt.ylabel("Normalised value")
plt.title("Monod model vs data — glucose only")
plt.legend()
plt.tight_layout()
plt.show()

print("Questions:")
print("  1. Do both curves have the same general shape?")
print("  2. Which model phase corresponds to the exponential phase?")
print("  3. What would happen if we compared the model with diauxie curves?")

### Growth rate $\mu$

In exponential phase, OD doubles at regular intervals.
Does the doubling speed — the **growth rate** $\mu$ (in h⁻¹) —
depend on glucose concentration?

In log scale, the exponential phase appears as a **straight line**
whose slope is exactly $\mu$.

> **Note**: this measurement uses the logarithm function (`np.log`),
> covered in 12th grade. Simply remember: *steeper slope = faster growth*.


In [ ]:
# ── Measure µ: slope of log(OD) vs time in exponential phase ─────────────────────
# (np.log = logarithm function — positive slope: growth, zero slope: stationary)

OD_min_exp = 0.02    # début de la phase exponentielle  ADAPTER si nécessaire
OD_max_exp = 0.30    # fin de la phase exponentielle    ADAPTER si nécessaire

mu_vals = []
puit_ok = []

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for well in wells_glu:
    if puit not in od_df.columns:
        continue
    od_corr = od_df[puit] - od_blanc
    mask = (od_corr > OD_min_exp) & (od_corr < OD_max_exp)
    if mask.sum() < 4:
        print(f"  {puit}: not enough points in exponential phase — skip")
        continue

    t_exp  = od_df['temps_h'][mask].values
    od_exp = od_corr[mask].values

    # Linear fit on log(OD): log(OD) = µ * t + constant
    coeffs = np.polyfit(t_exp, np.log(od_exp), 1)
    mu = coeffs[0]
    mu_vals.append(mu)
    puit_ok.append(puit)

    axes[0].plot(od_df['temps_h'], od_corr, lw=1.5, label=well)
    od_fit = np.exp(coeffs[1] + coeffs[0] * t_exp)  # np.exp: inverse of log, covered in 12th grade
    axes[0].plot(t_exp, od_fit, '--', color='gray', lw=1.2, alpha=0.8)

    od_pos = od_corr[od_corr > 0.005]
    axes[1].plot(od_df['temps_h'][od_corr > 0.005],
                 np.log(od_pos), lw=1.5, label=f'{well}  µ={mu:.2f} h⁻¹')

axes[0].set_xlabel("Time (hours)")
axes[0].set_ylabel("OD600")
axes[0].set_title("Growth curves with exponential fit")
axes[0].legend()
axes[1].set_xlabel("Time (hours)")
axes[1].set_ylabel("log(OD)")
axes[1].set_title("Log scale — slope = µ")
axes[1].legend()

plt.tight_layout()
plt.show()

print("Measured growth rates:")
for puit, mu in zip(puit_ok, mu_vals):
    t_double_min = 0.693 / mu * 60   # 0.693 ≈ ln(2) — temps de doublement en minutes
    print(f"  {puit}: µ = {mu:.2f} h⁻¹  →  doubling in ~{t_double_min:.0f} min")
print()
print("-> Does µ vary with [glucose]? Monod predicts µ ≈ µ_max if [glucose] >> Ks.")

### Diauxie: growth on a nutrient mixture

We will overlay the **glucose-only** and **glucose + lactose** growth curves
to identify the characteristic diauxie pause.

#### Diauxie — recap

When *E. coli* grows on glucose + lactose, it first consumes glucose
(preferred source), then **pauses** while inducing the lactose enzymes
(lacZ, lacY), before resuming growth on lactose. This double growth
gives its name to *diauxie* (from Greek *twice*) — and explains why
the *lac* operon is regulated!

> **To observe**: does the height of the first plateau depend on [glucose]?
> Does the final height depend on [lactose]?


In [ ]:
# ── Diauxie layout — ADAPT ───────────────────────────────────────────────────────
wells_diauxie = ['B1', 'B2']    # glucose + lactose

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Left: linear ────────────────────────────────────────────────────────────────
for well in wells_glu:
    if well in od_df.columns:
        axes[0].plot(od_df['temps_h'], od_df[puit] - od_blanc,
                     '--', lw=1.2, alpha=0.7, label=f'{well} (glu seul)')
for well in wells_diauxie:
    if well in od_df.columns:
        axes[0].plot(od_df['temps_h'], od_df[puit] - od_blanc,
                     '-', lw=2, label=f'{well} (glu+lac)')
axes[0].set_xlabel("Time (hours)")
axes[0].set_ylabel("OD600 (blank-corrected)")
axes[0].set_title("Diauxie vs glucose only — linear")
axes[0].legend(fontsize=9)

# ── Right: log — the diauxie pause appears as a dip ─────────────────────────────
for well in wells_diauxie:
    if puit not in od_df.columns:
        continue
    od_corr = od_df[puit] - od_blanc
    mask = od_corr > 0.01
    axes[1].plot(od_df['temps_h'][mask], np.log(od_corr[mask]),
                 '-', lw=2, label=well)
axes[1].set_xlabel("Time (hours)")
axes[1].set_ylabel("log(OD)")
axes[1].set_title("Diauxie — log scale")
axes[1].legend()

plt.tight_layout()
plt.show()

print("Questions:")
print("  1. Linear scale: can you see two distinct growth phases?")
print("  2. Log scale: the dip corresponds to the lacZ induction pause.")
print("     During this pause, what is the bacterium doing?")
print("  3. Final OD (glu+lac) > final OD (glu only)? By how much?")
print("     Is this consistent with N_max = N0 + Y*(S_glu + S_lac)?")

---
## Going further

1. **Modify parameters**: what happens biologically if `p_max` increases?
   If `Ks` decreases? Which one is easier to modify experimentally?

2. **Two strains**: simulate two populations with different `p_max` sharing
   the same medium (same $S$ reservoir). Which one dominates at the end of the culture? Why?

3. **Doubling time**: from your experimental data, estimate the
   doubling time in exponential phase.
   *Hint*: it is the slope of the line in log scale.

4. **Variability**: compare the spread of trajectories between the
   exponential phase (small N) and stationary phase (large N). Why does it change?
